**Table of contents**<a id='toc00_'></a> 
- [0. Imports](#toc0_)
    - [0.1. Packages and Libraries](#toc0_1_)
    - [0.2. Data](#toc0_2_)
- [1. Base Table Updates](#toc1_)
    - [1.1. Taxonomy](#toc1_1_)
    - [1.2. Realms](#toc1_2_)
    - [1.3. References](#toc1_3_)
- [2. Data Info](#toc2_)
    - [2.1. Observations](#toc2_1_)
    - [2.2. Native Data](#toc2_2_)
- [3. Tables Preparation](#toc3_)
    - [3.1. Native Data](#toc3_1_)
        - [3.1.1. Most Recent Equal Combinations](#toc3_1_1_)
        - [3.1.2. Update Table With Keys](#toc3_1_2_)
            - [3.1.2.1. Realm](#toc3_1_2_1_)
            - [3.1.2.2. Taxonomy](#toc3_1_2_2_)
            - [3.1.2.3. References](#toc3_1_2_3_)
    - [3.2. Observations Data](#toc3_2_)
        - [3.2.1. Update Table With Keys](#toc3_2_1_)
            - [3.2.1.1. Area - Geographical Region](#toc_3_2_1_1_)
            - [3.2.1.2. Realm](#toc_3_2_1_2_)
            - [3.2.1.3. Taxonomy](#toc_3_2_1_3_)
            - [3.2.1.4. References](#toc_3_2_1_4_)
- [4. Confirm Tables](#toc4_)
    - [4.1. Taxonomy](#toc4_1_)
    - [4.2. Area - Geographical Region](#toc4_2_)
    - [4.3. Realm](#toc4_3_)
    - [4.4. References](#toc4_4_)
    - [4.5. Natives](#toc4_5_)
    - [4.6. Observations](#toc4_6_)
- [5. Database Tables Export](#toc5_)
- [6. SQLite3 Database Creation](#toc6_)

# <a id='toc0_'></a> [0. Imports](#toc00_)

## <a id='toc0_1_'></a> [0.1. Packages and Libraries](#toc0_)

In [1]:
import pandas as pd
import sqlite3

## <a id='toc0_2_'></a> [0.2. Data](#toc0_)

In [2]:
RawData = pd.read_csv(r'../Data Raw/ObsList.csv')
RawData.drop_duplicates(inplace=True)
TaxonomyData = pd.read_csv(r'../Data Raw/TaxonomyRaw.csv')
TaxonomyData.drop_duplicates(inplace=True)
Regions = pd.read_csv(r'../Data Raw/RegionsTableData.csv', sep=';')
Regions.drop_duplicates(inplace=True)
Natives = pd.read_csv(r'../Data Raw/NativeRaw.csv', sep=';')
Natives.drop_duplicates(inplace=True)


In [3]:
RawData.head(5)

,Species,Area,Realm,Cryptogenic,Introduced,Dispersal,Established,Eradicated,Intentional_Release,Year,Reference Year,Reference
0,Spodoptera eridania,Brazil,Neotropical,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
1,Phyllocnistis citrella,Brazil,Neotropical,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
2,Hypsipyla grandella,Brazil,Neotropical,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
3,Spodoptera eridania,United States,Nearctic,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...
4,Hyphantria cunea,United States,Nearctic,NaN,NaN,NaN,1.0,0.0,NaN,NaN,2024,CAB International (CABI) (2024). CABI Invasive...


In [4]:
TaxonomyData.head(5)

,Species,AcceptedSpecies,Genus,Family
0,Spodoptera eridania,Spodoptera eridania,Spodoptera,Noctuidae
1,Phyllocnistis citrella,Phyllocnistis citrella,Phyllocnistis,Gracillariidae
2,Hypsipyla grandella,Hypsipyla grandella,Hypsipyla,Pyralidae
3,Hyphantria cunea,Hyphantria cunea,Hyphantria,Erebidae
4,Phthorimaea operculella,Phthorimaea operculella,Phthorimaea,Gelechiidae


In [5]:
Regions.head(5)

,AreaID,AreaName,Country,Continent,SubContinent
0,AFG.1_1,Afghanistan,Afghanistan,Asia,NaN
1,AGO.1_1,Angola,Angola,Africa,NaN
2,ALB.1_1,Albania,Albania,Europe,NaN
3,AND.1_1,Andorra,Andorra,Europe,NaN
4,ARE.1_1,United Arab Emirates,United Arab Emirates,Asia,NaN


In [6]:
Natives.head(5)

,Species,AcceptedSpecies,Continent,Realm,Cosmopolitan,Reference Year,Reference
0,Bleszynskia malacelloides,Bleszynskia malacelloides,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
1,Artigisa melanephele,Artigisa melanephele,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
2,Acrocercops laciniella,Acrocercops laciniella,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
3,Zomariana doxasticana,Zomariana doxasticana,Australia,Australian,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
4,Coleophora striatipennella,Coleophora striatipennella,Europe,Palearctic,0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."


# <a id='toc1_'></a> [1. Base Tables Updates](#toc00_)

Besides the Areas (Geographical Regions) base table there were no tables fully udpdated on the database, so we started by separating the data that was fully merged on the observation table onto separate parts. Composing: Taxonomy, Realms and References.

## <a id='toc1_1_'></a> [1.1. Taxonomy](#toc1_)

In [7]:
TaxonomyData['SpeciesID'] = 'SP' + (TaxonomyData.index +1).astype(str)

In [8]:
species_to_id = dict(zip(TaxonomyData['Species'], TaxonomyData['SpeciesID']))
TaxonomyData['AcceptedSpeciesID'] = TaxonomyData['AcceptedSpecies'].map(species_to_id)

In [9]:
missing_mask = TaxonomyData['AcceptedSpeciesID'].isnull()
missing_species = TaxonomyData[missing_mask]['AcceptedSpecies'].unique()

new_rows = []
for species in missing_species:
    ref_row = TaxonomyData[TaxonomyData['AcceptedSpecies'] == species].iloc[0]
    new_rows.append({
        'Species': species,
        'AcceptedSpecies': species,
        'Genus': ref_row['Genus'],
        'Family': ref_row['Family']})

if new_rows:
    new_df = pd.DataFrame(new_rows)
    TaxonomyData = pd.concat([TaxonomyData, new_df], ignore_index=True)

TaxonomyData['SpeciesID'] = 'SP' + (TaxonomyData.index + 1).astype(str)

species_to_id = dict(zip(TaxonomyData['Species'], TaxonomyData['SpeciesID']))
TaxonomyData['AcceptedSpeciesID'] = TaxonomyData['AcceptedSpecies'].map(species_to_id)


In [10]:
TaxonomyData.head(5)

,Species,AcceptedSpecies,Genus,Family,SpeciesID,AcceptedSpeciesID
0,Spodoptera eridania,Spodoptera eridania,Spodoptera,Noctuidae,SP1,SP1
1,Phyllocnistis citrella,Phyllocnistis citrella,Phyllocnistis,Gracillariidae,SP2,SP2
2,Hypsipyla grandella,Hypsipyla grandella,Hypsipyla,Pyralidae,SP3,SP3
3,Hyphantria cunea,Hyphantria cunea,Hyphantria,Erebidae,SP4,SP4
4,Phthorimaea operculella,Phthorimaea operculella,Phthorimaea,Gelechiidae,SP5,SP5


## <a id='toc1_2_'></a> [1.2. Realms](#toc1_)

In [11]:
RealmObs = RawData[['Realm']].copy()
RealmNat = Natives[['Realm']].copy()

Realms = pd.concat([RealmObs, RealmNat]).drop_duplicates().reset_index(drop=True)
Realms['RealmID'] = 'RLM' + (Realms.index +1).astype(str)

In [12]:
Realms

,Realm,RealmID
0,Neotropical,RLM1
1,Nearctic,RLM2
2,Oceanina,RLM3
3,Oriental,RLM4
4,Sino-Japanese,RLM5
5,Afrotropical,RLM6
6,Palearctic,RLM7
7,Australian,RLM8
8,Panamanian,RLM9
9,Saharo-Arabian,RLM10


## <a id='toc1_3_'></a> [1.3. References](#toc1_)

In [13]:
ReferencesObs = RawData[['Reference Year', 'Reference']].copy()
ReferencesObs.drop_duplicates(inplace=True)
ReferencesObs.reset_index(drop=True, inplace=True)

In [14]:
ReferencesObs.head(5)

,Reference Year,Reference
0,2024,CAB International (CABI) (2024). CABI Invasive...
1,2020,"Gilligan, T., Brown, J. and Baixeras, J. (2020..."
2,2024,European and Mediterranean Plant Protection Or...
3,2009,"Fodor, E. and Haruta, O. (2009). Niche partiti..."
4,2010,"Heard, T., Elliott, L., Anderson, B., White, L..."


In [15]:
ReferencesNat = Natives[['Reference Year', 'Reference']].copy()
ReferencesNat.drop_duplicates(inplace=True)
ReferencesNat.reset_index(drop=True, inplace=True)

In [16]:
ReferencesNat

,Reference Year,Reference
0,2001,"Hoare, R. (2001). Adventive species of Lepidop..."
1,1992,"Frank, J. and McCoy, E. (1992). Introduction t..."
2,2010,"Lopez-Vaamonde, C., Agassiz, D., Augustin, S.,..."
3,2020,"Gilligan, T., Brown, J. and Baixeras, J. (2020..."
4,2022,"Calhoun, J. and Smith, R. (2022). Brephidium e..."
...,...,...
720,1960,Gozmany L. (1960). The results of the zoologic...
721,2017,"Cock, M. (2017) A preliminary catalogue of the..."
722,1987,"Diakonoff, A. and van Nieukerken, E. (1987). E..."
723,2003,"Baran, T. (2003). Scythris buszkoi sp. n., a n..."


In [17]:
References = pd.concat([ReferencesObs, ReferencesNat]).drop_duplicates().reset_index(drop=True)


In [18]:
References['ReferenceID'] = 'REF' + (References.index +1).astype(str)

In [19]:
References

,Reference Year,Reference,ReferenceID
0,2024,CAB International (CABI) (2024). CABI Invasive...,REF1
1,2020,"Gilligan, T., Brown, J. and Baixeras, J. (2020...",REF2
2,2024,European and Mediterranean Plant Protection Or...,REF3
3,2009,"Fodor, E. and Haruta, O. (2009). Niche partiti...",REF4
4,2010,"Heard, T., Elliott, L., Anderson, B., White, L...",REF5
...,...,...,...
1100,2023,"Assaad, M. (2023). Ecology and impacts of inse...",REF1101
1101,1960,Gozmany L. (1960). The results of the zoologic...,REF1102
1102,2017,"Cock, M. (2017) A preliminary catalogue of the...",REF1103
1103,1987,"Diakonoff, A. and van Nieukerken, E. (1987). E...",REF1104


# <a id='toc2_'></a> [2. Data Info](#toc00_)

## <a id='toc2_1_'></a> [2.1. Observations](#toc2_)

In [20]:
RawData.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16505 entries, 0 to 21198
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Species              16505 non-null  object 
 1   Area                 16505 non-null  object 
 2   Realm                16505 non-null  object 
 3   Cryptogenic          4434 non-null   float64
 4   Introduced           7177 non-null   float64
 5   Dispersal            647 non-null    float64
 6   Established          16305 non-null  float64
 7   Eradicated           16461 non-null  float64
 8   Intentional_Release  4596 non-null   float64
 9   Year                 3057 non-null   float64
 10  Reference Year       16505 non-null  int64  
 11  Reference            16505 non-null  object 
dtypes: float64(7), int64(1), object(4)
memory usage: 1.6+ MB


In [21]:
RawData.describe().T

,count,mean,std,min,25%,50%,75%,max
Cryptogenic,4434.0,0.143888,0.351016,0.0,0.0,0.0,0.0,1.0
Introduced,7177.0,0.970183,0.170095,0.0,1.0,1.0,1.0,1.0
Dispersal,647.0,0.701700,0.457866,0.0,0.0,1.0,1.0,1.0
Established,16305.0,0.968353,0.175063,0.0,1.0,1.0,1.0,1.0
Eradicated,16461.0,0.003888,0.062234,0.0,0.0,0.0,0.0,1.0
Intentional_Release,4596.0,0.054830,0.227673,0.0,0.0,0.0,0.0,1.0
Year,3057.0,1974.614001,51.200735,1565.0,1958.0,1991.0,2008.0,2024.0
Reference Year,16505.0,2020.437201,6.376645,1926.0,2020.0,2024.0,2024.0,2025.0


In [22]:
RawData.describe(include = 'O').T

,count,unique,top,freq
Species,16505,1616,Helicoverpa armigera,362
Area,16505,233,United States,1015
Realm,16505,11,Palearctic,7272
Reference,16505,503,European and Mediterranean Plant Protection Or...,4657


In [23]:
RawData.columns

Index(['Species', 'Area', 'Realm', 'Cryptogenic', 'Introduced', 'Dispersal',
       'Established', 'Eradicated', 'Intentional_Release', 'Year',
       'Reference Year', 'Reference'],
      dtype='object')

## <a id='toc2_2_'></a> [2.2. Native Data](#toc2_)

In [24]:
Natives.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2546 entries, 0 to 7749
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Species          2546 non-null   object
 1   AcceptedSpecies  2546 non-null   object
 2   Continent        2544 non-null   object
 3   Realm            2546 non-null   object
 4   Cosmopolitan     2546 non-null   int64 
 5   Reference Year   2546 non-null   int64 
 6   Reference        2546 non-null   object
dtypes: int64(2), object(5)
memory usage: 159.1+ KB


In [25]:
Natives.describe().T

,count,mean,std,min,25%,50%,75%,max
Cosmopolitan,2546.0,0.001571,0.039614,0.0,0.0,0.0,0.0,1.0
Reference Year,2546.0,2006.905342,20.125541,1775.0,2003.0,2011.0,2020.0,2025.0


In [26]:
Natives.describe(include = 'O').T

,count,unique,top,freq
Species,2546,920,Lampides boeticus,14
AcceptedSpecies,2546,909,Lampides boeticus,14
Continent,2544,7,Asia,888
Realm,2546,11,Palearctic,807
Reference,2546,725,"Khramov, P. (Ed.) (2007). Insecta.pro: interna...",154


In [27]:
Natives.columns

Index(['Species', 'AcceptedSpecies', 'Continent', 'Realm', 'Cosmopolitan',
       'Reference Year', 'Reference'],
      dtype='object')

# <a id='toc3_'></a> [3. Tables Preparation](#toc00_)

Before we start inputing data onto the database we need to prepare the tables for all of them to have the predefined structure. We needed to update the fields for it to have the connections with the ID's. 

## <a id='toc3_1_'></a> [3.1. Native Data](#toc3_)

In [28]:
Natives_DB = Natives[['Species', 'AcceptedSpecies', 'Continent', 'Realm', 'Cosmopolitan', 'Reference Year', 'Reference']].copy()

### <a id='toc3_1_1_'></a> [3.1.1. Most Recent Equal Combinations](#toc3_1_)

In [29]:
idx = Natives_DB.groupby(['AcceptedSpecies', 'Continent', 'Realm', 'Cosmopolitan'])['Reference Year'].idxmax()

Natives_DB = Natives_DB.loc[idx].reset_index(drop=True)


### <a id='toc3_1_2_'></a> [3.1.2. Update Table with Keys](#toc3_1_)

#### <a id='toc3_1_2_1_'></a> [3.1.2.1. Realm](#toc3_1_2_)

In [30]:
Natives_DB = Natives_DB.merge(Realms, on='Realm', how='left')
Natives_DB.drop(columns='Realm', inplace=True)

#### <a id='toc3_1_2_2_'></a> [3.1.2.2. Taxonomy](#toc3_1_2_)

In [31]:
Natives_DB.drop(columns='AcceptedSpecies', inplace=True)
Natives_DB = Natives_DB.merge(TaxonomyData, on='Species', how='left')

In [32]:
Natives_DB.drop(columns=['Species', 'AcceptedSpecies', 'Genus', 'Family', 'AcceptedSpeciesID'], inplace=True)

#### <a id='toc3_1_2_3_'></a> [3.1.2.3. References](#toc3_1_2_)

In [33]:
Natives_DB.drop(columns='Reference Year', inplace=True)
Natives_DB = Natives_DB.merge(References, on='Reference', how='left')
Natives_DB.drop(columns=['Reference', 'Reference Year'], inplace=True)

## <a id='toc3_2_'></a> [3.2. Observations Data](#toc3_)

In [34]:
Observations_DB = RawData[['Species', 'Area', 'Realm', 'Cryptogenic', 'Introduced', 'Dispersal',
       'Established', 'Eradicated', 'Intentional_Release', 'Year',
       'Reference Year', 'Reference']].copy()

### <a id='toc3_2_1_'></a> [3.2.1. Update Table With Keys](#toc3_2_)


#### <a id='toc3_2_1_1_'></a> [3.2.1.1. Area - Geographical Regions](#toc3_2_1_)

In [35]:
RegionsData = Regions[['AreaID', 'AreaName', 'Country', 'Continent', 'SubContinent']].copy()

In [36]:
Observations_DB = Observations_DB.merge(RegionsData, left_on='Area', right_on='AreaName', how='left')
Observations_DB.drop(columns=['AreaName', 'Area', 'Country', 'Continent', 'SubContinent'], inplace=True)

#### <a id='toc3_2_1_2_'></a> [3.2.1.2. Realm](#toc3_2_1_)

In [37]:
Observations_DB = Observations_DB.merge(Realms, on='Realm', how='left')
Observations_DB.drop(columns='Realm', inplace=True)

#### <a id='toc3_2_1_3_'></a> [3.2.1.3. Taxonomy](#toc3_2_1_)

In [38]:
Observations_DB = Observations_DB.merge(TaxonomyData, on='Species', how='left')
Observations_DB.drop(columns=['Species', 'AcceptedSpecies', 'Genus', 'Family', 'AcceptedSpeciesID'], inplace=True)

#### <a id='toc3_2_1_4_'></a> [3.2.1.4. References](#toc3_2_1_)

In [39]:
Observations_DB.drop(columns='Reference Year', inplace=True)
Observations_DB = Observations_DB.merge(References, on='Reference', how='left')
Observations_DB.drop(columns=['Reference', 'Reference Year'], inplace=True)

# <a id='toc4_'></a> [4. Confirm Tables](#toc00_)

## <a id='toc4_1_'></a> [4.1. Taxonomy](#toc00_)

In [40]:
TaxonomyData.head(5)

,Species,AcceptedSpecies,Genus,Family,SpeciesID,AcceptedSpeciesID
0,Spodoptera eridania,Spodoptera eridania,Spodoptera,Noctuidae,SP1,SP1
1,Phyllocnistis citrella,Phyllocnistis citrella,Phyllocnistis,Gracillariidae,SP2,SP2
2,Hypsipyla grandella,Hypsipyla grandella,Hypsipyla,Pyralidae,SP3,SP3
3,Hyphantria cunea,Hyphantria cunea,Hyphantria,Erebidae,SP4,SP4
4,Phthorimaea operculella,Phthorimaea operculella,Phthorimaea,Gelechiidae,SP5,SP5


In [41]:
TaxonomyData.columns

Index(['Species', 'AcceptedSpecies', 'Genus', 'Family', 'SpeciesID',
       'AcceptedSpeciesID'],
      dtype='object')

In [42]:
TaxonomyData.describe().T

,count,unique,top,freq
Species,1705,1705,Spodoptera eridania,1
AcceptedSpecies,1705,1395,Neogalea sunia,4
Genus,1705,813,Coleophora,29
Family,1705,75,Tortricidae,204
SpeciesID,1705,1705,SP1,1
AcceptedSpeciesID,1705,1395,SP512,4


In [43]:
TaxonomyData.describe(include = 'O').T

,count,unique,top,freq
Species,1705,1705,Spodoptera eridania,1
AcceptedSpecies,1705,1395,Neogalea sunia,4
Genus,1705,813,Coleophora,29
Family,1705,75,Tortricidae,204
SpeciesID,1705,1705,SP1,1
AcceptedSpeciesID,1705,1395,SP512,4


## <a id='toc4_2_'></a> [4.2. Area - Geographical Region](#toc00_)

In [44]:
RegionsData.head(5)

,AreaID,AreaName,Country,Continent,SubContinent
0,AFG.1_1,Afghanistan,Afghanistan,Asia,NaN
1,AGO.1_1,Angola,Angola,Africa,NaN
2,ALB.1_1,Albania,Albania,Europe,NaN
3,AND.1_1,Andorra,Andorra,Europe,NaN
4,ARE.1_1,United Arab Emirates,United Arab Emirates,Asia,NaN


In [45]:
RegionsData.columns

Index(['AreaID', 'AreaName', 'Country', 'Continent', 'SubContinent'], dtype='object')

In [46]:
RegionsData.describe().T

,count,unique,top,freq
AreaID,258,258,AFG.1_1,1
AreaName,258,241,Spain,4
Country,258,205,France,8
Continent,258,8,Africa,64
SubContinent,22,3,Polynesia,9


In [47]:
RegionsData.describe(include = 'O').T

,count,unique,top,freq
AreaID,258,258,AFG.1_1,1
AreaName,258,241,Spain,4
Country,258,205,France,8
Continent,258,8,Africa,64
SubContinent,22,3,Polynesia,9


## <a id='toc4_3_'></a> [4.3. Realms](#toc00_)

In [48]:
Realms.head(5)

,Realm,RealmID
0,Neotropical,RLM1
1,Nearctic,RLM2
2,Oceanina,RLM3
3,Oriental,RLM4
4,Sino-Japanese,RLM5


In [49]:
Realms.columns

Index(['Realm', 'RealmID'], dtype='object')

In [50]:
Realms.describe().T

,count,unique,top,freq
Realm,11,11,Neotropical,1
RealmID,11,11,RLM1,1


In [51]:
Realms.describe(include = 'O').T

,count,unique,top,freq
Realm,11,11,Neotropical,1
RealmID,11,11,RLM1,1


## <a id='toc4_4_'></a> [4.4. References](#toc00_)

In [52]:
References.head(5)

,Reference Year,Reference,ReferenceID
0,2024,CAB International (CABI) (2024). CABI Invasive...,REF1
1,2020,"Gilligan, T., Brown, J. and Baixeras, J. (2020...",REF2
2,2024,European and Mediterranean Plant Protection Or...,REF3
3,2009,"Fodor, E. and Haruta, O. (2009). Niche partiti...",REF4
4,2010,"Heard, T., Elliott, L., Anderson, B., White, L...",REF5


In [53]:
References.columns

Index(['Reference Year', 'Reference', 'ReferenceID'], dtype='object')

In [54]:
References.describe().T

,count,mean,std,min,25%,50%,75%,max
Reference Year,1105.0,2005.130317,21.593939,1775.0,2000.0,2012.0,2019.0,2025.0


In [55]:
References.describe(include = 'O').T

,count,unique,top,freq
Reference,1105,1105,CAB International (CABI) (2024). CABI Invasive...,1
ReferenceID,1105,1105,REF1,1


## <a id='toc4_5_'></a> [4.5. Natives](#toc00_)

In [56]:
Natives_DB.head(5)

,Continent,Cosmopolitan,RealmID,SpeciesID,ReferenceID
0,North America,0,RLM2,SP405,REF505
1,North America,0,RLM9,SP405,REF505
2,Africa,0,RLM10,SP1015,REF148
3,Asia,0,RLM7,SP1015,REF506
4,Asia,0,RLM10,SP1015,REF506


In [57]:
Natives_DB.columns

Index(['Continent', 'Cosmopolitan', 'RealmID', 'SpeciesID', 'ReferenceID'], dtype='object')

In [58]:
Natives_DB.describe().T

,count,mean,std,min,25%,50%,75%,max
Cosmopolitan,2200.0,0.001818,0.042611,0.0,0.0,0.0,0.0,1.0


In [59]:
Natives_DB.describe(include = 'O').T

,count,unique,top,freq
Continent,2200,7,Asia,769
RealmID,2200,11,RLM7,634
SpeciesID,2200,914,SP146,14
ReferenceID,2200,646,REF1,106


## <a id='toc4_6_'></a> [4.6. Observations](#toc00_)

In [60]:
Observations_DB.head(5)

,Cryptogenic,Introduced,Dispersal,Established,Eradicated,Intentional_Release,Year,AreaID,RealmID,SpeciesID,ReferenceID
0,NaN,NaN,NaN,1.0,0.0,NaN,NaN,BRA.1_1,RLM1,SP1,REF1
1,NaN,NaN,NaN,1.0,0.0,NaN,NaN,BRA.1_1,RLM1,SP2,REF1
2,NaN,NaN,NaN,1.0,0.0,NaN,NaN,BRA.1_1,RLM1,SP3,REF1
3,NaN,NaN,NaN,1.0,0.0,NaN,NaN,USA.1_1,RLM2,SP1,REF1
4,NaN,NaN,NaN,1.0,0.0,NaN,NaN,USA.1_2,RLM2,SP1,REF1


In [61]:
Observations_DB.columns

Index(['Cryptogenic', 'Introduced', 'Dispersal', 'Established', 'Eradicated',
       'Intentional_Release', 'Year', 'AreaID', 'RealmID', 'SpeciesID',
       'ReferenceID'],
      dtype='object')

In [62]:
Observations_DB.describe().T

,count,mean,std,min,25%,50%,75%,max
Cryptogenic,6421.0,0.137985,0.344911,0.0,0.0,0.0,0.0,1.0
Introduced,9062.0,0.972302,0.164115,0.0,1.0,1.0,1.0,1.0
Dispersal,929.0,0.719053,0.449704,0.0,0.0,1.0,1.0,1.0
Established,20418.0,0.968949,0.173460,0.0,1.0,1.0,1.0,1.0
Eradicated,20715.0,0.003862,0.062026,0.0,0.0,0.0,0.0,1.0
Intentional_Release,6863.0,0.047210,0.212103,0.0,0.0,0.0,0.0,1.0
Year,4360.0,1970.705046,58.216410,1565.0,1953.0,1989.0,2007.0,2024.0


In [63]:
Observations_DB.describe(include = 'O').T

,count,unique,top,freq
AreaID,18726,232,USA.1_2,1015
RealmID,20774,11,RLM7,8985
SpeciesID,20774,1616,SP23,399
ReferenceID,20774,503,REF3,5506


# <a id='toc5_'></a> [5. Database Tables Exportation](#toc00_)

In [64]:
TaxonomyData.to_csv(r'../Database Tables/Base_Taxonomy.csv', index=False)
RegionsData.to_csv(r'../Database Tables/Geography_Regions.csv', index=False)
Realms.to_csv(r'../Database Tables/Geography_Realms.csv', index=False)
References.to_csv(r'../Database Tables/Base_References.csv', index=False)
Natives_DB.to_csv(r'../Database Tables/Obs_NativesDB.csv', index=False)
Observations_DB.to_csv(r'../Database Tables/Obs_ObservationsDB.csv', index=False)

# <a id='toc6_'></a> [6. SQLite3 Database Creation](#toc00_)

## <a id='toc6_1_'></a> [6.1. Create Connection](#toc00_)

In [65]:
con = sqlite3.connect("../Database/ENNLWD.db")
con.execute("PRAGMA foreign_keys = ON")

In [66]:
cur = con.cursor()

## <a id='toc6_1_'></a> [6.2. Taxonomy](#toc00_)

In [67]:
TaxonomyData = TaxonomyData.reset_index(drop=True)

In [68]:
cur.execute("""CREATE TABLE IF NOT EXISTS Taxonomy (
    SpeciesID VARCHAR(7) PRIMARY KEY, 
    AcceptedSpeciesID VARCHAR(7),
    Species TEXT, 
    AcceptedSpecies TEXT, 
    Genus TEXT, 
    Family TEXT)
    """)

In [69]:
con.commit()

In [70]:
TaxonomyData.to_sql("Taxonomy", con, if_exists="append", index=False)

1705

In [71]:
res = cur.execute("SELECT name FROM sqlite_master")
res.fetchone()

('Taxonomy',)

In [ ]:
#cur.execute("""DROP TABLE Taxonomy""")

In [73]:
#cur.execute("CREATE TABLE Taxonomy(SpeciesID, AcceptedSpeciesID, Species, AcceptedSpecies, Genus, Family)")

# 6.X Close Connection

In [74]:
con.close()